# Week 2 — Data Audit & Cleaning
### E-Commerce Order Fulfillment Dataset (50K Records)
**Yuva Internship — Logistics Track**

**Objective:** Audit the raw order fulfillment dataset for quality issues (missing values, duplicates, inconsistent formats, outliers) and produce a clean, analysis-ready dataset (`orders_clean.csv`) for the Week 3 EDA phase.

**Script reference:** `scripts/01_data_audit_cleaning.py`
**Input:** `data/E-Commerce_Order_Fulfillment_Dataset_50K_Records.csv`
**Output:** `data/orders_clean.csv`


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load Raw Dataset

In [ ]:
RAW_PATH = "data/E-Commerce_Order_Fulfillment_Dataset_50K_Records.csv"
CLEAN_PATH = "data/orders_clean.csv"

df = pd.read_csv(RAW_PATH)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()


## 3. Initial Inspection

Check column names, data types, and get a first look at the structure of the dataset.

In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


## 4. Missing Value Audit

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_report


**Handling strategy:**
- Categorical fields (e.g. `Customer_Region`, `Product_Category`, `Shipping_Mode`) → fill with `"Unknown"` rather than dropping rows.
- Numeric fields (e.g. `Shipping_Cost`) → fill with the column median (robust to outliers).
- Date fields (`Order_Date`, `Ship_Date`, `Delivery_Date`) → rows with missing dates are flagged; dropped only if the record cannot be reasonably reconstructed.
- `Delivery_Status` missing → inferred from `Delivery_Date` presence where possible, else marked `"Unknown"`.

In [ ]:
categorical_cols = ['Customer_Region', 'Product_Category', 'Shipping_Mode', 'Delivery_Status']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

if 'Shipping_Cost' in df.columns:
    df['Shipping_Cost'] = df['Shipping_Cost'].fillna(df['Shipping_Cost'].median())

print("Remaining missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])


## 5. Duplicate Records

In [ ]:
dupe_count = df.duplicated().sum()
print(f"Fully duplicated rows: {dupe_count}")

if 'Order_ID' in df.columns:
    dupe_ids = df.duplicated(subset=['Order_ID']).sum()
    print(f"Duplicate Order_IDs: {dupe_ids}")

df = df.drop_duplicates()
if 'Order_ID' in df.columns:
    df = df.drop_duplicates(subset=['Order_ID'], keep='first')

print(f"Rows after de-duplication: {len(df):,}")


## 6. Standardize Text / Categorical Fields

Trim whitespace and normalize casing so the same category isn't split by formatting differences (e.g. `" express"` vs `"Express"`).

In [ ]:
text_cols = ['Customer_Region', 'Product_Category', 'Shipping_Mode', 'Delivery_Status']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

for col in text_cols:
    if col in df.columns:
        print(col, "->", df[col].unique()[:10])


## 7. Date Fields — Parsing & Consistency Checks

In [ ]:
date_cols = ['Order_Date', 'Ship_Date', 'Delivery_Date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Logical sequence check: Order_Date <= Ship_Date <= Delivery_Date
bad_sequence = df[(df['Ship_Date'] < df['Order_Date']) | (df['Delivery_Date'] < df['Ship_Date'])]
print(f"Rows with illogical date sequence: {len(bad_sequence)}")


In [ ]:
# Drop rows where dates are impossible (delivery before shipping, shipping before order)
df = df[~((df['Ship_Date'] < df['Order_Date']) | (df['Delivery_Date'] < df['Ship_Date']))]
print(f"Rows remaining after date-sequence cleanup: {len(df):,}")


## 8. Recompute / Validate `Delivery_Days`

Recalculate `Delivery_Days` from `Ship_Date` and `Delivery_Date` and compare against the original column to catch data-entry errors.

In [ ]:
df['Delivery_Days_Calc'] = (df['Delivery_Date'] - df['Ship_Date']).dt.days

if 'Delivery_Days' in df.columns:
    mismatch = df[df['Delivery_Days'] != df['Delivery_Days_Calc']]
    print(f"Rows where stored Delivery_Days disagrees with calculated value: {len(mismatch)}")
    # Trust the recalculated value as the source of truth
    df['Delivery_Days'] = df['Delivery_Days_Calc']
else:
    df['Delivery_Days'] = df['Delivery_Days_Calc']

df = df.drop(columns=['Delivery_Days_Calc'])


## 9. Outlier Detection (IQR method)

Applied to `Shipping_Cost` and `Delivery_Days` — flag rather than blindly delete, since some 'outliers' (e.g. long international deliveries) may be legitimate.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Shipping_Cost', 'Delivery_Days']:
    if col in df.columns:
        low, high = iqr_bounds(df[col])
        outliers = df[(df[col] < low) | (df[col] > high)]
        print(f"{col}: bounds=({low:.2f}, {high:.2f}) | outliers flagged={len(outliers)}")


In [ ]:
# Remove records with clearly invalid values (negative cost or negative delivery days)
before = len(df)
if 'Shipping_Cost' in df.columns:
    df = df[df['Shipping_Cost'] >= 0]
if 'Delivery_Days' in df.columns:
    df = df[df['Delivery_Days'] >= 0]
print(f"Rows removed for invalid negative values: {before - len(df)}")


## 10. Final Validation

In [ ]:
print("Final shape:", df.shape)
print("\nRemaining nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nRemaining duplicates:", df.duplicated().sum())
df.head()


## 11. Export Cleaned Dataset

In [ ]:
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEAN_PATH}")
print(f"Final record count: {len(df):,}")


## 12. Cleaning Summary (fill in after running on the real dataset)

| Metric | Value |
|---|---|
| Raw record count | *(fill in)* |
| Missing values fixed | *(fill in)* |
| Duplicate rows removed | *(fill in)* |
| Rows removed (bad date sequence) | *(fill in)* |
| Rows removed (negative values) | *(fill in)* |
| Final clean record count | *(fill in)* |

**Next step (Week 3):** Exploratory Data Analysis & Visualization on `data/orders_clean.csv` — delivery performance, shipping cost distribution, regional and shipping-mode breakdowns.